In [ ]:
from itertools import product
import sys
sys.path.append('/home/projects/nyosef/zvise/PixelGen/')
from PixelGen.multimodalvi import MultiModalSCVI
from PixelGen.multimodalvae import MultiModalVAE, AggMethod, D
from PixelGen.enums import AggMethod, D
from PixelGen.metrics import MultiModalVIMetrics
from sklearn.preprocessing import PowerTransformer

from rich import print
import anndata as ad
import pixelator
import torch
import scvi
import scipy
# from scvi import autotune

import seaborn as sns
import scanpy as sc
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from tqdm import tqdm

from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection

# import ray
# from ray import tune


from PixelGen.pxl_utils import train_model, get_model_latents
from PixelGen.scvi_utils import plot_losses, pca_neighbors_umap, calc_PCA
from pixelator.common.statistics import clr_transformation, dsb_normalize


from pixelator.pna.plot import molecule_rank_plot
# from pixelator.plot import molecule_rank_plot, cell_count_plot, scatter_umi_per_upia_vs_tau
# from pixelator.statistics import clr_transformation
# from pixelator.analysis.normalization import dsb_normalize


from sklearn.preprocessing import StandardScaler, MinMaxScaler 

from PixelGen.pxl_utils import train_model, get_model_latents, convert_polarization_to_feature_matrix, \
     convert_colocalization_to_feature_matrix, download_pxl
from PixelGen.scvi_utils import plot_losses, pca_neighbors_umap, calc_PCA, add_one_hot_encoding_obsm, plot_cumulative_variance
from PixelGen.common_utils import standardize, std_clip, filter_hv, split_pair_column, filter_df_by_two_columns, rank_plot
from PixelGen.metrics import MultiModalVIMetrics, distr_autocorrelation_in_latent
from PixelGen.multimodalvi import MultiModalSCVI
from PixelGen.multimodalvae import MultiModalVAE, AggMethod, D
from PixelGen.enums import AggMethod, D

import tempfile

from scvi import REGISTRY_KEYS
from scvi.module.base import (
    BaseModuleClass,
    LossOutput,
    PyroBaseModuleClass,
    auto_move_data,
)
from torch.distributions import NegativeBinomial, Normal, Poisson, MixtureSameFamily, Beta
from torch.distributions import kl_divergence as kl



# from cytovi import CytoVI

print(torch.cuda.is_available())
from sklearn.decomposition import PCA


scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)
sc.set_figure_params(figsize=(6, 6), frameon=False)
sns.set_theme()
torch.set_float32_matmul_precision("high")
save_dir = tempfile.TemporaryDirectory()

%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"
%load_ext autoreload
%autoreload 2

In [ ]:
def spatial_hvg(adata,obsm_name,n_top_genes=159):
    protein_matrix = adata.obsm[obsm_name]

    protein_names = protein_matrix.columns if hasattr(protein_matrix, 'columns') else [f"protein_{i}" for i in range(protein_matrix.shape[1])]

    # Create temporary AnnData for HVG selection
    adata_tmp = sc.AnnData(
        X=protein_matrix.values if hasattr(protein_matrix, 'values') else protein_matrix,
        obs=adata.obs.copy(),
        var=pd.DataFrame(index=protein_names)
    )

    # Run HVG selection
    sc.pp.highly_variable_genes(
        adata_tmp,
        flavor="seurat",
        n_top_genes=n_top_genes
    )

    # Subset to HV proteins
    hv_mask = adata_tmp.var["highly_variable"]
    adata_hvg = adata_tmp[:, hv_mask].copy()
    
    return adata_hvg.X

def spatial_pca(adata, obsm_name, n_components=159):
    protein_matrix = adata.obsm[obsm_name]
    X = protein_matrix.values if hasattr(protein_matrix, 'values') else protein_matrix

    # Run PCA
    pca = PCA(n_components=n_components)
    X_pca = pca.fit_transform(X)
    return X_pca

In [ ]:
ADATA_PATH='/home/projects/nyosef/zvise/PixelGen/PixelGen/adata_final_features.h5ad'
adata_annotated=sc.read('/home/projects/nyosef/zvise/PixelGen/PixelGen/adata_annotated.h5ad')

adata=ad.read_h5ad(ADATA_PATH)
adata.obs = pd.merge(
    adata.obs,
    adata_annotated.obs[['cell_type']],
    left_index=True,
    right_index=True,
    how='left'
)
adata

In [ ]:
all_abundance_layers=[
    obsm for obsm in adata.obsm.keys() if obsm.startswith('abund')
]

abundance_layers=['abundance_arcsinh5']
for a in abundance_layers:
    adata.layers[a]=adata.obsm[a].copy()
    print (adata.obsm[a].shape)

print (abundance_layers)
print (adata.layers)

In [ ]:
all_spatial_layers=[
    obsm for obsm in adata.obsm.keys() if obsm.startswith('spz')
]

spatial_layers=['spz_asinh3__DR__MORAN_TOPK__PCA']

for obsm_name in spatial_layers:
    df= pd.DataFrame(
    adata.obsm[obsm_name],
    index=adata.obs_names)
    adata.obsm[obsm_name]=df
    print (adata.obsm[obsm_name].shape)
    
    print (obsm_name)

In [ ]:
adata.obsm['pca'] = PCA(n_components=30).fit_transform(adata.X)


In [ ]:
num=0
layer=spatial_layers[num]

plt.hist(adata.obsm[layer].iloc[:, 1], bins=30)
plt.xlabel('PC1 Value')
plt.ylabel('Frequency')
plt.title('Histogram of First PCA Component (spatial)')
plt.show()

## PARATER TUNING

In [ ]:
models_dict={}

In [ ]:
model_cls = MultiModalSCVI

def run_abundance_model(ab_layer, model_name, max_epocs=10000):
    print(f"Running Abundance-only model {model_name}")
        
    latent_name=f'{model_name}_latent'
    
    setup_kwargs = dict(layer=ab_layer, n_modalities=1, batch_key=None, )
    model_kwargs = dict(n_latent=30, n_hidden=128, n_layers=1, dropout_rate=0.1, 
                            distrs=[D.Normal,], 
                            
                            loss_weights='auto',
                            
                            external_kl_weight=1,
                            decoder_kwargs=dict(decoder_param_eps=1e-2, decoder_activation='exp')
                        )
    train_kwargs = dict(train_size=0.8, check_val_every_n_epoch=1, early_stopping=True, 
                        early_stopping_patience=200, batch_size=2000,
                        max_epochs=max_epocs, enable_checkpointing=True, 
                        plan_kwargs=dict(lr=1e-4, optimizer='Adam', n_epochs_kl_warmup=400)
                    )
    model = train_model(adata, model_cls=model_cls, setup_kwargs=setup_kwargs, model_kwargs=model_kwargs, train_kwargs=train_kwargs,)

    modalities_latent_names=[('joint', latent_name)]
    get_model_latents(adata, model, modalities_latent_names=modalities_latent_names)

    return model

def run_spatial_model(sp_layer, model_name,max_epochs=10000):
    print(f"Running spatial-only model {model_name}")
        
    latent_name=f'{model_name}_latent'
    
    setup_kwargs = dict(layer=sp_layer, n_modalities=1, batch_key=None, )
    model_kwargs = dict(n_latent=30, n_hidden=128, n_layers=1, dropout_rate=0.1, 
                            distrs=[D.Normal,], 
                            
                            loss_weights='auto',
                            
                            external_kl_weight=1,
                            decoder_kwargs=dict(decoder_param_eps=1e-2, decoder_activation='exp')
                        )
    train_kwargs = dict(train_size=0.8, check_val_every_n_epoch=1, early_stopping=True, 
                        early_stopping_patience=200, batch_size=2000,
                        max_epochs=max_epochs, enable_checkpointing=True, 
                        plan_kwargs=dict(lr=1e-4, optimizer='Adam', n_epochs_kl_warmup=400)
                    )
    model = train_model(adata, model_cls=model_cls, setup_kwargs=setup_kwargs, model_kwargs=model_kwargs, train_kwargs=train_kwargs,)

    modalities_latent_names=[('joint', latent_name)]
    get_model_latents(adata, model, modalities_latent_names=modalities_latent_names)

    return model

def run_joined_embed(abundance_layer, spatial_layer, model_name, max_epochs=10000):
    
    print(f"Running joined_embed model {model_name}")
        
    latent_name=f'{model_name}_latent'
    
    setup_kwargs = dict(layer=abundance_layer, extra_modality_keys=[spatial_layer], n_modalities=2, batch_key=None, )
    model_kwargs = dict(n_latent=30, n_hidden=128, n_layers=2, dropout_rate=0.1, 
                            distrs=[D.Normal, D.Normal], 
                            agg_method=AggMethod.SHARED_ENCODER,
                            loss_weights='auto',
                            joint_kl=True, unimodal_kl=False,
                            external_kl_weight=1,
                            decoder_kwargs=dict(decoder_param_eps=1e-2, decoder_activation='exp')
                        )
    train_kwargs = dict(train_size=0.8, check_val_every_n_epoch=1, early_stopping=True, 
                        early_stopping_patience=200, batch_size=2000,
                        max_epochs=max_epochs, enable_checkpointing=True, 
                        plan_kwargs=dict(lr=1e-4, optimizer='Adam', n_epochs_kl_warmup=400)
                    )
    model = train_model(adata, model_cls=model_cls, setup_kwargs=setup_kwargs, model_kwargs=model_kwargs, train_kwargs=train_kwargs,)

    modalities_latent_names=[('joint', latent_name)]
    get_model_latents(adata, model, modalities_latent_names=modalities_latent_names)
    return model
    
def run_weight_join(abundance_layer, spatial_layer, model_name, max_epochs=10000):
    
    print(f"Running weight model {model_name}")
        
    latent_name=f'{model_name}_latent'
    
    setup_kwargs = dict(layer=abundance_layer, extra_modality_keys=[spatial_layer], n_modalities=2, batch_key=None, )
    model_kwargs = dict(n_latent=30, n_hidden=128, n_layers=2, dropout_rate=0.1, 
                            distrs=[D.Normal, D.Normal], 
                            
                            loss_weights='auto',
                            joint_kl=False, unimodal_kl=True,
                            
                            decoder_kwargs=dict(decoder_param_eps=1e-2, decoder_activation='exp')
                        )
    train_kwargs = dict(train_size=0.8, check_val_every_n_epoch=1, early_stopping=True, 
                        early_stopping_patience=200, batch_size=2000,
                        max_epochs=max_epochs, enable_checkpointing=True, 
                        plan_kwargs=dict(lr=1e-4, optimizer='Adam', n_epochs_kl_warmup=400)
                    )
    model = train_model(adata, model_cls=model_cls, setup_kwargs=setup_kwargs, model_kwargs=model_kwargs, train_kwargs=train_kwargs,)


    modalities_latent_names=[(s,f'{s}_latent') for s in ('joint',abundance_layer, spatial_layer)]
    get_model_latents(adata, model, modalities_latent_names=modalities_latent_names)
    return model    

## ABUNDANCE ONLY

In [ ]:
abundance_layers

In [ ]:
max_epochs=10000

#  Abundance-only
for a in abundance_layers:
    if a is not None:   # skip None if you only want actual preprocessing
        model_name=f'abn_arcsinh'
        model = run_abundance_model(a, model_name, max_epocs=max_epochs)
        models_dict[model_name]=model
        


In [ ]:
metrics=MultiModalVIMetrics(
    adata,
    models_dict,
    pca_key='pca'
)

metrics.run()

In [ ]:
_ = metrics.mean_modality_errors_barplot(reconstruction_mean=True)

In [ ]:
metrics.plot_negative_likelihood()

In [ ]:
index_dict=metrics.plot_latent_comparison_metrics()
print("Latent index mapping:")
for k, v in index_dict.items():
        print(f"  {k} → {v}")

In [ ]:
df = metrics.plot_latent_quality_metrics()
display (df)

In [ ]:
metrics.plot_scib_metrics()

## SPATIAL ONLY

In [ ]:
# spatial_layer_fits=['spz_id__filtered__ALL','spz_tanh4__filtered__ALL','spz_asinh3__filtered__HVGs','spz_ranknorm_cell__filtered__MORAN_TOPK']
# for o in spatial_layer_fits:
#     adata.layers[o]=adata.obsm[o].copy()

# #  spatial-only
# for spatial_layer in spatial_layer_fits:
#     if spatial_layer is not None:  
#         model_name=f'spatial_only_{spatial_layer}'
#         model = run_spatial_model(spatial_layer, model_name, max_epochs=max_epochs)
#         models_dict[model_name]=model




## Joint encoder

In [ ]:
#  joined embedding
models_dict = {}

for abundance_layer, spatial_layer in product(abundance_layers, spatial_layers):
    model_name=f'joined_arcinh'
    model=run_joined_embed(abundance_layer, spatial_layer, model_name, max_epochs=max_epochs)
    models_dict[model_name]=model
    

In [ ]:
metrics=MultiModalVIMetrics(
    adata,
    models_dict,
    pca_key='pca'
)

metrics.run()

In [ ]:
_ = metrics.mean_modality_errors_barplot(reconstruction_mean=True)

In [ ]:
metrics.plot_negative_likelihood()

In [ ]:
index_dict=metrics.plot_latent_comparison_metrics()
print("Latent index mapping:")
for k, v in index_dict.items():
        print(f"  {k} → {v}")

In [ ]:
df = metrics.plot_latent_quality_metrics()
display (df)

In [ ]:
metrics.plot_scib_metrics()

## WEIGHTED MODEL

In [ ]:
# # weighted embeed
# models_dict = {}

for abundance_layer, spatial_layer in product(abundance_layers, spatial_layers):
    model_name='weighted_arcsinh'
    model=run_weight_join(abundance_layer, spatial_layer, model_name, max_epochs=max_epochs)
    models_dict[model_name]=model

## METRICS

In [ ]:
metrics=MultiModalVIMetrics(
    adata,
    models_dict,
    pca_key='pca'
)

metrics.run()

In [ ]:
_ = metrics.mean_modality_errors_barplot(reconstruction_mean=True)

In [ ]:
metrics.plot_negative_likelihood()

In [ ]:
index_dict=metrics.plot_latent_comparison_metrics()
print("Latent index mapping:")
for k, v in index_dict.items():
        print(f"  {k} → {v}")

In [ ]:
df = metrics.plot_latent_quality_metrics()
display (df)

In [ ]:
metrics.plot_scib_metrics()

## FiNAL

In [ ]:
abundance_layer_parameters=['dsb']
spatial_layer_parameters=['preproces_spatial_arcsinh_dim_pca']

max_epochs=10000
models_dict = {}

for a in abundance_layer_parameters:    
    model_name=f'abundance_only_{a}'
    model = run_abundance_model(a, model_name, max_epocs=max_epochs)
    models_dict[model_name]=model

for spatial_layer in spatial_layer_parameters:

    model_name=f'spatial_only_{spatial_layer}'
    model = run_spatial_model(spatial_layer, model_name, max_epochs=max_epochs)
    models_dict[model_name]=model
    
for abundance_layer, spatial_layer in product(abundance_layer_parameters, spatial_layer_parameters):
    model_name=f'joined_embed_abun_{abundance_layer}_spatial_{spatial_layer}'
    model=run_joined_embed(abundance_layer, spatial_layer, model_name, max_epochs=max_epochs)
    models_dict[model_name]=model
    
for abundance_layer, spatial_layer in product(abundance_layer_parameters, spatial_layer_parameters):
    model_name=f'weighted_embed_abun_{abundance_layer}_spatial_{spatial_layer}'
    model=run_weight_join(abundance_layer, spatial_layer, model_name, max_epochs=max_epochs)
    models_dict[model_name]=model

In [ ]:
metrics=MultiModalVIMetrics(
    adata,
    models_dict,
    pca_key='pca'
)

metrics.run()

In [ ]:
_ = metrics.mean_modality_errors_barplot()
_ = metrics.mean_modality_errors_barplot(reconstruction_mean=True)

In [ ]:
_ = metrics.mean_autocorr_barplot()


In [ ]:
adata

In [ ]:
best_obsm='joined_embed_abun_dsb_spatial_preproces_spatial_arcsinh_dim_pca_latent'

In [ ]:

sc.pp.neighbors(adata, use_rep=best_obsm)


sc.tl.umap(adata)

sc.tl.leiden(adata, resolution=0.8, flavor="igraph", n_iterations=2)

sc.pl.umap(adata, color=['leiden','cell_group','condition'])

In [ ]:
adata.obsm["preproces_spatial_arcsinh_dim_hvg"] = np.asarray(
    adata.obsm["preproces_spatial_arcsinh_dim_hvg"], dtype=np.float32
)

adata.obsm["preproces_spatial_arcsinh_dim_pca"] = np.asarray(
    adata.obsm["preproces_spatial_arcsinh_dim_pca"], dtype=np.float32
)

adata.obsm["preproces_spatial_dim_hvg"] = np.asarray(
    adata.obsm["preproces_spatial_dim_hvg"], dtype=np.float32
)

adata.obsm["preproces_spatial_dim_pca"] = np.asarray(
    adata.obsm["preproces_spatial_dim_pca"], dtype=np.float32
)


adata.write_h5ad('/home/projects/nyosef/zvise/PixelGen/PixelGen/adata_umap_latent.h5ad')

In [ ]:
best_obsm='joined_embed_abun_dsb_spatial_preproces_spatial_dim_pca_latent'

In [ ]:

sc.pp.neighbors(adata, use_rep=best_obsm)


sc.tl.umap(adata)

sc.tl.leiden(adata, resolution=0.8, flavor="igraph", n_iterations=2)

sc.pl.umap(adata, color=['leiden','cell_group','condition'])